In [ ]:
print(df.columns.tolist())

# 常见清洗步骤示例

In [23]:
# 安装依赖（仅在环境未安装时运行）
!pip install pandas sqlalchemy pymysql matplotlib seaborn

In [26]:
import pandas as pd
from sqlalchemy import create_engine

# 填写本地数据库连接信息
LOCAL_USER = 'datax'
LOCAL_PWD  = 'DxPwd123!'
LOCAL_HOST = 'BPeril'
LOCAL_PORT = 3306
LOCAL_DB   = 'DIDatax'

engine = create_engine(f"mysql+pymysql://{LOCAL_USER}:{LOCAL_PWD}@{LOCAL_HOST}:{LOCAL_PORT}/{LOCAL_DB}?charset=utf8mb4")


# 示例：读取 ga_session 表（请根据需要修改 SQL）
df = pd.read_sql('SELECT * FROM ga_session WHERE date BETWEEN "201702" AND "201705"', engine)

In [21]:
print(df.columns.tolist())

['uid', 'full_visitor_id', 'user_id', 'visit_number', 'visit_id', 'visit_start_time', 'date', 'social_engagement_type', 'channel_grouping']


In [27]:
# 修复后的清洗步骤：仅保留实际存在的列
# 1) 查看表结构与缺失值
print("DataFrame info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

# 2) 去重
df = df.drop_duplicates()

# 3) 字段筛选（根据实际存在的列自动适配）
# 目标保留列（包含可能不存在的列）
cols_keep_target = ['uid','full_visitor_id','visit_number','visit_start_time','date','channel_grouping', 'browser', 'deviceCategory']

# 自动只选存在的列，避免 KeyError
cols_exist = [c for c in cols_keep_target if c in df.columns]
print(f"\n保留的列: {cols_exist}")

df_small = df[cols_exist].copy()

# 4) 缺失值处理（仅当列存在时处理）
if 'browser' in df_small.columns:
    df_small['browser'] = df_small['browser'].fillna('unknown')

# 5) 保存为 CSV
output_path = '../data/ga_session_2017_02_05_cleaned.csv'
# 确保目录存在
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_small.to_csv(output_path, index=False)
print(f"\n清洗完成，已保存至 {output_path}")

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199175 entries, 0 to 199174
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   uid                     199175 non-null  object
 1   full_visitor_id         199175 non-null  object
 2   user_id                 0 non-null       object
 3   visit_number            199175 non-null  int64 
 4   visit_id                199175 non-null  int64 
 5   visit_start_time        199175 non-null  int64 
 6   date                    199175 non-null  object
 7   social_engagement_type  199175 non-null  object
 8   channel_grouping        199175 non-null  object
dtypes: int64(3), object(6)
memory usage: 13.7+ MB
None

Missing values:
user_id                   199175
uid                            0
full_visitor_id                0
visit_number                   0
visit_id                       0
visit_start_time               0
date                